1. Create a table, make 3 changes to it (insert, update, insert), and use DESCRIBE HISTORY to review the
resulting versions.

In [0]:
-- Step 1: Create a table
CREATE TABLE IF NOT EXISTS cyntexa_dev.default.demo_versions (
  id INT,
  name STRING,
  value INT
);

-- Step 2: First insert
INSERT INTO cyntexa_dev.default.demo_versions VALUES (1, 'Alice', 100), (2, 'Bob', 200);

-- Step 3: Update a row
UPDATE cyntexa_dev.default.demo_versions SET value = 150 WHERE id = 1;

-- Step 4: Second insert
INSERT INTO cyntexa_dev.default.demo_versions VALUES (3, 'Charlie', 300);

-- Step 5: Review the version history
DESCRIBE HISTORY cyntexa_dev.default.demo_versions;

2. Use COPY INTO to incrementally load 2 batches of files into a bronze table, confirming COPY INTO
doesn't reprocess the first batch.

In [0]:
-- CREATE TABLE IF NOT EXISTS cyntexa_dev.bronze.orders_bronze (
--     order_id INT,
--     customer_id INT,
--     transaction_id INT,
--     product_id INT,
--     quantity INT,
--     discount_amount DOUBLE,
--     total_amount DOUBLE,
--     order_date DATE
-- )
-- USING DELTA;


In [0]:

COPY INTO cyntexa_dev.bronze.orders_bronze
FROM '/Volumes/cyntexa_dev/bronze/raw/sales/'
FILEFORMAT = CSV
FORMAT_OPTIONS (
    'header' = 'true',
    'inferSchema' = 'true'
);

In [0]:
-- SELECT * FROM cyntexa_dev.bronze.orders_bronze;
-- SELECT *
-- FROM read_files(
--     '/Volumes/cyntexa_dev/bronze/raw/sales/',
--     format => 'csv',
--     header => true,
--     inferSchema => true
-- )
-- limit 2;

In [0]:
-- DESCRIBE TABLE cyntexa_dev.bronze.orders_bronze;

In [0]:
DESCRIBE HISTORY cyntexa_dev.bronze.orders_bronze;

3. Query an old version of the table with both VERSION AS OF and TIMESTAMP AS OF.


In [0]:
%python
df_version = spark.read.format("delta").option("versionAsOf", 3).table("cyntexa_dev.bronze.orders_bronze")

df_timestamp = spark.read.format("delta").option("timestampAsOf","2026-08-27T10:52:21.000+00:00").table("cyntexa_dev.bronze.orders_bronze")

In [0]:
SELECT * FROM cyntexa_dev.bronze.orders_bronze VERSION AS OF 3;
SELECT * FROM cyntexa_dev.bronze.orders_bronze TIMESTAMP AS OF "2026-08-27T10:52:21.000+00:00";